# PiL-HQUC — GPU Demo and Offline Benchmark Suite

This notebook runs the delivered project with the final architecture:

- **Localhost demo:** Hybrid-only, Qamomile → CUDA-Q, NVIDIA GPU required.
- **Benchmark 1:** SimBench-derived Hybrid vs full HiGHS MILP.
- **Benchmark 2:** fixed 10-generator synthetic qubit scaling, q = 8–26.
- **Benchmark 3:** synthetic generator scaling, q = 10 and q = 20.
- One immutable quantum profile is used everywhere: depth 1, 256 final shots, 64 fallback optimizer shots, 6 COBYLA evaluations, and at most 2 ADMM-guided rounds.

Use a Colab GPU runtime before continuing: **Runtime → Change runtime type → GPU**.

In [ ]:
# 1. Upload and extract the delivered project ZIP safely.
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import files

os.chdir('/content')
WORK_DIR = Path('/content/pil_hquc_project')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

print('Upload the delivered Quantathon project ZIP.')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise RuntimeError('No ZIP file was uploaded.')

zip_path = Path('/content') / zip_names[0]
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(WORK_DIR)

candidates = [
    path for path in WORK_DIR.rglob('*')
    if path.is_dir() and (path / 'backend/app').exists() and (path / 'frontend').exists()
]
if not candidates:
    raise RuntimeError('Could not find a project root containing backend/app and frontend.')
ROOT = min(candidates, key=lambda path: len(path.parts))
os.chdir('/content')
print('Project root:', ROOT)
print('Current working directory:', os.getcwd())

In [ ]:
# 2. Install Python and frontend dependencies into the interpreter used by benchmark subprocesses.
import os
import subprocess
import sys
from pathlib import Path

PYTHON_BIN = '/usr/local/bin/python' if Path('/usr/local/bin/python').exists() else sys.executable
print('Benchmark Python:', PYTHON_BIN)

subprocess.run([PYTHON_BIN, '-m', 'pip', 'install', '--upgrade', 'pip'], cwd='/content', check=True)
subprocess.run(
    [PYTHON_BIN, '-m', 'pip', 'install', '-r', str(ROOT / 'benchmark/requirements-benchmark.txt')],
    cwd='/content',
    check=True,
)
subprocess.run([PYTHON_BIN, '-m', 'pip', 'check'], cwd='/content', check=True)
subprocess.run(['npm', 'ci'], cwd=ROOT / 'frontend', check=True)
print('Dependencies installed.')

In [ ]:
# 3. Verify that the exact runtime used by the project sees the NVIDIA CUDA-Q target.
import os
import subprocess

ENV = os.environ.copy()
ENV['CUDAQ_TARGET'] = 'nvidia'
ENV['REQUIRE_CUDAQ'] = '1'
ENV['PYTHONPATH'] = f"{ROOT / 'backend'}:{ROOT}"

check_code = r"""
import cudaq
cudaq.set_target('nvidia')
target = cudaq.get_target()
name = getattr(target, 'name', str(target))
name = name() if callable(name) else str(name)
print('CUDA-Q target:', name)
assert 'nvidia' in name.lower(), name
"""
subprocess.run([PYTHON_BIN, '-c', check_code], cwd='/content', env=ENV, check=True)
print('GPU requirement verified.')

In [ ]:
# 4. Run CPU-side unit tests before starting the GPU demo.
import subprocess

subprocess.run(
    [PYTHON_BIN, '-m', 'pytest', '-q', str(ROOT / 'backend/tests')],
    cwd=ROOT,
    env=ENV,
    check=True,
)

## Start the GPU localhost demo

The browser talks to the Vite server. Vite proxies `/api` to FastAPI, and FastAPI executes Qamomile → CUDA-Q on the Colab NVIDIA GPU. Keep this notebook runtime alive while using the demo.

In [ ]:
# 5. Start FastAPI and Vite in the background, then open the Colab proxy URL.
import os
import subprocess
import time
import urllib.request
from IPython.display import HTML, display
from google.colab import output

for process in globals().get('SERVER_PROCESSES', []):
    if process.poll() is None:
        process.terminate()
SERVER_PROCESSES = []

backend_log = open('/content/pil_hquc_backend.log', 'w')
frontend_log = open('/content/pil_hquc_frontend.log', 'w')

backend_process = subprocess.Popen(
    [PYTHON_BIN, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=ROOT / 'backend',
    env=ENV,
    stdout=backend_log,
    stderr=subprocess.STDOUT,
)
frontend_process = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '5173'],
    cwd=ROOT / 'frontend',
    env=ENV,
    stdout=frontend_log,
    stderr=subprocess.STDOUT,
)
SERVER_PROCESSES = [backend_process, frontend_process]

time.sleep(6)
health = urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=20).read().decode()
print('Backend health:', health)

demo_url = output.eval_js('google.colab.kernel.proxyPort(5173)')
display(HTML(f'<a href="{demo_url}" target="_blank" style="font-size:18px">Open PiL-HQUC GPU Demo</a>'))
print('Backend log: /content/pil_hquc_backend.log')
print('Frontend log: /content/pil_hquc_frontend.log')

## Run the benchmark suite

`--quick` reduces only the number of seeds/data points. It **does not** change depth, shots, optimizer evaluations, Top-K rule, or maximum ADMM rounds.

In [ ]:
# 6. Quick end-to-end protocol check.
import subprocess

subprocess.run(
    [PYTHON_BIN, str(ROOT / 'benchmark/run_all.py'), '--experiments', 'all', '--quick'],
    cwd=ROOT,
    env=ENV,
    check=True,
)

In [ ]:
# 7. Full benchmark for the final report. This may take a long time at 24–26 qubits.
# Run this cell only when you are ready for the complete 3-seed experiment set.
import subprocess

RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    subprocess.run(
        [PYTHON_BIN, str(ROOT / 'benchmark/run_all.py'), '--experiments', 'all'],
        cwd=ROOT,
        env=ENV,
        check=True,
    )
else:
    print('Set RUN_FULL_BENCHMARK = True to run the complete benchmark.')

In [ ]:
# 8. Serve and open benchmark_report.html inside Colab.
import subprocess
import time
from IPython.display import HTML, display
from google.colab import output

old_report_server = globals().get('REPORT_SERVER')
if old_report_server is not None and old_report_server.poll() is None:
    old_report_server.terminate()

REPORT_SERVER = subprocess.Popen(
    [PYTHON_BIN, '-m', 'http.server', '8081', '--directory', str(ROOT / 'benchmark')],
    cwd='/content',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)
report_base = output.eval_js('google.colab.kernel.proxyPort(8081)')
report_url = f"{report_base}/report/benchmark_report.html"
display(HTML(f'<a href="{report_url}" target="_blank" style="font-size:18px">Open Benchmark Report</a>'))
print('Report file:', ROOT / 'benchmark/report/benchmark_report.html')

In [ ]:
# 9. Download the HTML report and a ZIP containing raw data, summaries and figures.
import shutil
from pathlib import Path
from google.colab import files

bundle_dir = Path('/content/pil_hquc_benchmark_bundle')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir()
shutil.copytree(ROOT / 'benchmark/results', bundle_dir / 'results')
shutil.copytree(ROOT / 'benchmark/report', bundle_dir / 'report')
archive = shutil.make_archive('/content/pil_hquc_benchmark_bundle', 'zip', root_dir=bundle_dir)

files.download(str(ROOT / 'benchmark/report/benchmark_report.html'))
files.download(archive)